# LECTURE 15 — HANDS-ON EXERCISE
### MODULE 6: DEEP LEARNING APPLICATIONS IN MACROECONOMICS

**HANDS-ON EXERCISE** — Forecast Exchange Rates Using a Recurrent Neural Network (LSTM)

*WAIFEM ML Facilitation Programme 2026 — Compatible with Google Colab & VS Code*

## ── SECTION 1: IMPORTS

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120})

try:
    import google.colab          # noqa: F401
    print("Environment : Google Colab")
except ImportError:
    print("Environment : Local (VS Code / terminal)")

print("Libraries loaded successfully.\n")

## ── SECTION 2: SYNTHETIC EXCHANGE RATE DATA (provided)

In [ ]:
def generate_exchange_rate_data(n_obs: int = 240, seed: int = 99) -> pd.DataFrame:
    """
    Generate 240 monthly observations (20 years) of synthetic exchange rate data.

    Features (X)  : inflation_diff, interest_rate_diff, gdp_growth,
                    trade_balance, commodity_index
    Target  (y)   : exchange_rate  (local currency per USD, log-differenced)
    """
    rng   = np.random.default_rng(seed)
    dates = pd.date_range(start='2004-01-01', periods=n_obs, freq='MS')
    t     = np.linspace(0, 1, n_obs)

    inflation_diff    = rng.normal(2.0, 1.5, n_obs)
    interest_rate_diff= rng.normal(3.0, 2.0, n_obs)
    gdp_growth        = 3.5 + 1.2 * np.sin(8 * np.pi * t) + rng.normal(0, 0.5, n_obs)
    trade_balance     = -2.0 + 1.5 * np.sin(4 * np.pi * t) + rng.normal(0, 1.0, n_obs)
    commodity_index   = 100 + 15 * np.sin(6 * np.pi * t) + rng.normal(0, 5, n_obs)

    # Exchange rate as function of fundamentals (purchasing power parity proxy)
    er_change = (
        0.35 * inflation_diff
        + 0.25 * interest_rate_diff
        - 0.20 * gdp_growth
        - 0.15 * trade_balance
        - 0.02 * (commodity_index - 100)
        + rng.normal(0, 0.8, n_obs)
    )

    # Cumulate to get level series
    level = 100.0 * np.exp(np.cumsum(er_change / 100))

    return pd.DataFrame({
        'exchange_rate':     level,
        'inflation_diff':    inflation_diff,
        'interest_rate_diff':interest_rate_diff,
        'gdp_growth':        gdp_growth,
        'trade_balance':     trade_balance,
        'commodity_index':   commodity_index,
    }, index=dates)


df = generate_exchange_rate_data()
FEATURE_COLS = ['inflation_diff', 'interest_rate_diff', 'gdp_growth',
                'trade_balance', 'commodity_index']
TARGET_COL   = 'exchange_rate'

print(f"Dataset : {df.shape[0]} monthly obs  x  {df.shape[1]} variables")
print(df.describe().round(2).to_string())

# Visualise exchange rate
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Synthetic Exchange Rate & Macro Drivers', fontsize=13, fontweight='bold')
for ax, col in zip(axes.flatten(), df.columns):
    ax.plot(df.index, df[col], linewidth=1.2)
    ax.set_title(col.replace('_', ' ').title())
    ax.tick_params(axis='x', rotation=30)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('lecture15_exercise_raw_data.png', bbox_inches='tight')
plt.show()
print("Saved : lecture15_exercise_raw_data.png")

## ── SECTION 3: PREPROCESSING (provided)

In [ ]:
LOOKBACK = 12

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_sc = scaler_X.fit_transform(df[FEATURE_COLS].values)
y_sc = scaler_y.fit_transform(df[[TARGET_COL]].values).ravel()

print(f"\nLookback window : {LOOKBACK} months")
print(f"Input features  : {FEATURE_COLS}")

## ── SECTION 4: BUILD SEQUENCES  ◄─ YOUR TASK (TODO 1)

Create sliding-window sequences for the LSTM.

Implement the function  make_sequences(X_data, y_data, lookback)  that:

- Loops from index `lookback` to the end of the data

- At each step i, appends:

X_seq : X_data[i-lookback : i, :]   → shape (lookback, n_features)

y_seq : y_data[i]                    → scalar (next time step target)

- Returns np.array(X_seq), np.array(y_seq)

After implementing, create X_all and y_all, then split chronologically:

SPLIT = int(0.80 * len(X_all))

X_train, X_test = X_all[:SPLIT], X_all[SPLIT:]

y_train, y_test = y_all[:SPLIT], y_all[SPLIT:]

Print the shapes to verify.

In [ ]:
# WRITE YOUR CODE BELOW
# def make_sequences(X_data, y_data, lookback):
#     ...
#
# X_all, y_all = make_sequences(X_sc, y_sc, LOOKBACK)
# SPLIT = ...
# X_train, X_test = ...
# y_train, y_test = ...
# print(f"Sequence shapes — X : {X_all.shape}  |  y : {y_all.shape}")
# print(f"Train : {X_train.shape[0]}  |  Test : {X_test.shape[0]}")

## ── SECTION 5: BUILD THE LSTM MODEL  ◄─ YOUR TASK (TODO 2)

Build an LSTM model for single-step regression.

Suggested architecture:

LSTM(64, return_sequences=True,  input_shape=(LOOKBACK, n_features))

Dropout(0.20)

LSTM(32, return_sequences=False)

Dropout(0.20)

Dense(16, activation='relu')

Dense(1,  activation='linear')

Compile with:

optimizer = Adam(learning_rate=1e-3)

loss      = 'mse'

metrics   = ['mae']

Call model.summary() to display the architecture.

Hint: n_features = X_train.shape[2]

In [ ]:
# WRITE YOUR CODE BELOW
# N_FEAT = X_train.shape[2]
# model  = Sequential([...])
# model.compile(...)
# model.summary()

## ── SECTION 6: TRAIN THE MODEL  ◄─ YOUR TASK (TODO 3)

Train your model:

- EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

- epochs=200, batch_size=32, validation_split=0.15

- Store the history object for plotting learning curves.

In [ ]:
# WRITE YOUR CODE BELOW
# history = model.fit(...)

## ── SECTION 7: EVALUATE & VISUALISE  ◄─ YOUR TASK (TODO 4)

a) Plot learning curves (train vs val loss).

b) Generate predictions:

y_pred_sc = model.predict(X_test).ravel()

y_pred    = scaler_y.inverse_transform(y_pred_sc.reshape(-1,1)).ravel()

y_actual  = scaler_y.inverse_transform(y_test.reshape(-1,1)).ravel()

c) Compute RMSE, MAE, R² and print them.

d) Also build a Linear Regression baseline:

X_tr_flat = X_train.reshape(X_train.shape[0], -1)

X_te_flat = X_test.reshape(X_test.shape[0], -1)

lr = LinearRegression().fit(X_tr_flat, y_train)

y_lr = scaler_y.inverse_transform(lr.predict(X_te_flat).reshape(-1,1)).ravel()

e) Plot actual vs LSTM vs Linear on the same axes.

Use  test_dates = df.index[LOOKBACK + SPLIT:]  for the x-axis.

In [ ]:
# WRITE YOUR CODE BELOW

# ── Discussion Questions ──────────────────────────────────────────────────────
print("""
── Discussion Questions ───────────────────────────────────────────────────────
 1. How does the LSTM's R² compare to the linear baseline?
    What does this tell you about exchange rate predictability?

 2. Exchange rates are often described as a random walk.
    Does your model's performance support or challenge this view?

 3. Which feature (inflation differential, interest rate differential, etc.)
    do you think has the greatest predictive power? How would you test this?

 4. How would you adapt this model for a 3-month-ahead forecast?
──────────────────────────────────────────────────────────────────────────────
""")

print("=== EXERCISE COMPLETE ===")